# 04 — Gold | Star-Schema Serving Layer for Power BI

**Project:** MLB Performance Intelligence Platform

Exposes one clean star schema in the `gold` schema so BI tools never read Silver directly.

| Object | Type | Role in the model |
|---|---|---|
| `gold.dim_date` | table | Calendar (every date between first and last game) |
| `gold.dim_team`, `gold.dim_player`, `gold.dim_venue` | views | Conformed dimensions |
| `gold.fact_game` | view | One row per game, with team names and attendance/duration |
| `gold.fact_team_game` | view | Team perspective of each completed game |
| `gold.fact_batting_game`, `gold.fact_pitching_game` | views | Player lines per game |
| `gold.league_daily_summary` | table | Daily league KPIs (the v1 Gold table, corrected: only completed games count) |

Relationships for Power BI are documented in `powerbi/README.md`.

In [ ]:
import sys, importlib
_nb = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
sys.path.insert(0, "/Workspace" + _nb.rsplit("/", 2)[0] + "/shared")
import config as C
import pipeline as P
for _m in (C, P):
    importlib.reload(_m)

RUN_ID = P.new_run_id()
T = lambda s, n: C.table(s, n)

In [ ]:
with P.log_step(spark, RUN_ID, "gold", "dim_date") as step:
    dim_date = spark.sql(f"""
    WITH b AS (SELECT MIN(official_date) AS lo, MAX(official_date) AS hi FROM {T('silver', 'fact_game')})
    SELECT d AS date, YEAR(d) AS year, MONTH(d) AS month, DATE_FORMAT(d, 'MMMM') AS month_name,
           DATE_FORMAT(d, 'MMM yyyy') AS month_year, CAST(DATE_FORMAT(d, 'yyyyMM') AS INT) AS month_key,
           WEEKOFYEAR(d) AS week_of_year, DAYOFWEEK(d) AS day_of_week, DATE_FORMAT(d, 'EEEE') AS day_name,
           DAYOFWEEK(d) IN (1, 7) AS is_weekend
    FROM b LATERAL VIEW EXPLODE(SEQUENCE(MAKE_DATE(YEAR(b.lo), 1, 1), MAKE_DATE(YEAR(b.hi), 12, 31), INTERVAL 1 DAY)) AS d
    """)
    step.records = P.replace_table(spark, dim_date, T("gold", "dim_date"), "Gold: calendar dimension covering every season in the platform")

In [ ]:
def cols(table, alias=None, exclude=("updated_at",)):
    """Explicit column list (portable alternative to SELECT * EXCEPT)."""
    prefix = f"{alias}." if alias else ""
    return ", ".join(prefix + c for c in spark.table(table).columns if c not in exclude)

views = {
    "dim_team": ("Gold: team dimension", f"SELECT {cols(T('silver', 'dim_team'))} FROM {T('silver', 'dim_team')}"),
    "dim_player": ("Gold: player dimension (birth country, position, handedness)",
                   f"SELECT {cols(T('silver', 'dim_player'))} FROM {T('silver', 'dim_player')}"),
    "dim_venue": ("Gold: ballpark dimension", f"SELECT {cols(T('silver', 'dim_venue'))} FROM {T('silver', 'dim_venue')}"),
    "fact_game": ("Gold: one row per game with team names, attendance and duration", f"""
        SELECT {cols(T('silver', 'fact_game'), 'g', ('source_extracted_at', 'updated_at'))},
               h.team_abbreviation AS home_team, a.team_abbreviation AS away_team,
               CONCAT(a.team_abbreviation, ' @ ', h.team_abbreviation) AS matchup,
               d.attendance, d.duration_minutes, d.weather, d.home_plate_umpire
        FROM {T('silver', 'fact_game')} g
        LEFT JOIN {T('silver', 'dim_team')} h ON h.team_id = g.home_team_id
        LEFT JOIN {T('silver', 'dim_team')} a ON a.team_id = g.away_team_id
        LEFT JOIN {T('silver', 'game_details')} d ON d.game_pk = g.game_pk"""),
    "fact_team_game": ("Gold: completed games from each team perspective",
                       f"SELECT {cols(T('silver', 'fact_team_game'))} FROM {T('silver', 'fact_team_game')} WHERE is_completed"),
    "fact_batting_game": ("Gold: batting line per player per game",
                          f"SELECT {cols(T('silver', 'fact_batting_game'))} FROM {T('silver', 'fact_batting_game')}"),
    "fact_pitching_game": ("Gold: pitching line per appearance",
                           f"SELECT {cols(T('silver', 'fact_pitching_game'))} FROM {T('silver', 'fact_pitching_game')}"),
}
with P.log_step(spark, RUN_ID, "gold", "serving_views") as step:
    for name, (comment, sql) in views.items():
        spark.sql(f"CREATE OR REPLACE VIEW {T('gold', name)} COMMENT '{comment}' AS {sql}")
    step.records = len(views)

In [ ]:
league_daily = spark.sql(f"""
SELECT g.official_date AS game_date, g.season, g.season_type,
  COUNT(*) AS scheduled_games,
  SUM(INT(g.is_completed)) AS completed_games,
  SUM(INT(g.status_detail = 'Postponed')) AS postponed_games,
  SUM(INT(g.is_completed AND g.was_rescheduled)) AS makeup_games,
  SUM(g.total_runs) AS total_runs,
  ROUND(AVG(g.total_runs), 2) AS avg_runs_per_game,
  SUM(INT(g.is_extra_innings)) AS extra_inning_games,
  SUM(INT(g.is_one_run_game)) AS one_run_games,
  MAX(b.home_runs) AS home_runs,
  ROUND(AVG(d.attendance)) AS avg_attendance,
  ROUND(AVG(d.duration_minutes)) AS avg_duration_minutes
FROM {T('silver', 'fact_game')} g
LEFT JOIN {T('silver', 'game_details')} d ON d.game_pk = g.game_pk
LEFT JOIN (SELECT official_date, SUM(hr) AS home_runs FROM {T('silver', 'fact_batting_game')} GROUP BY official_date) b
       ON b.official_date = g.official_date
GROUP BY g.official_date, g.season, g.season_type
""")

with P.log_step(spark, RUN_ID, "gold", "league_daily_summary") as step:
    step.records = P.replace_table(spark, league_daily, T("gold", "league_daily_summary"),
        "Gold: daily league KPIs (completed and postponed games, runs, home runs, extra innings, attendance, duration)")

In [ ]:
display(spark.sql(f"SHOW TABLES IN {C.CATALOG}.gold"))